# ML Pipeline

Bu notebook fraud detection sürecini production-ready bir pipeline haline getiriyor. Neden önemli? Notebook'ta ayrı ayrı çalışan adımlar production'da birbirine bağlı değilse felaket kapıda. Training'de uygulanan bir preprocessing adımını inference'da unutursan, model tamamen farklı veri görür ve çöp sonuç üretir.

04_FeatureEngineering'de türettiğimiz 36 feature'dan SHAP analizi ile seçilen 34 tanesi ve 05_ModelOptimization'daki Optuna parametreleri tek bir sklearn Pipeline'ında birleşiyor. `pipeline.fit()` ile eğit, `pipeline.predict()` ile production'da kullan - arada hiçbir adım kaybolmaz.

In [1]:
import pandas as pd
import numpy as np
import warnings
import os
import json
import joblib

from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score, GroupKFold
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
np.random.seed(42)
print("Setup complete")

Setup complete


## 1. Veri ve Parametreler

In [2]:
data_dir = '../../data/processed'
model_dir = '../../models/fraud_detection'

X_train_full = pd.read_csv(f'{data_dir}/X_train.csv')
X_val_full = pd.read_csv(f'{data_dir}/X_val.csv')
y_train = pd.read_csv(f'{data_dir}/y_train.csv').squeeze()
y_val = pd.read_csv(f'{data_dir}/y_val.csv').squeeze()

# ============================================================
# FEATURE SELECTION: SHAP + Native Rank Sum Threshold = 66
# 06_ModelEvaluation'da belirlenen optimum threshold
# rank_sum > 66 olan feature'lar çıkarılıyor
# AUC: 0.8842 | Recall: 0.6513 | Precision: 0.2512 | F1: 0.3625
# ============================================================
# Çıkarılan feature'lar (rank_sum > 66):
excluded_features = ['addr2_freq', 'card3_freq']  # rank_sum: 69, 71
selected_features = [f for f in X_train_full.columns if f not in excluded_features]

X_train = X_train_full[selected_features]
X_val = X_val_full[selected_features]

print(f"Feature Selection (SHAP + Native Rank Sum <= 66):")
print(f"  Original: {X_train_full.shape[1]} features")
print(f"  Selected: {len(selected_features)} features")
print(f"  Excluded: {excluded_features}")

# Load TransactionDT for time-based validation (same as 05_ModelOptimization)
train_full = pd.read_csv('../../data/train_optimized.csv', usecols=['TransactionDT'])

# Create month_num (approximately 30 days per month)
train_full['month_num'] = (train_full['TransactionDT'] // (86400 * 30)).astype('int16')

# Align with X_train + X_val indices
month_train = train_full['month_num'].iloc[:len(X_train)].values
month_val = train_full['month_num'].iloc[len(X_train):len(X_train)+len(X_val)].values

with open(f'{model_dir}/optimized/optimization_metadata.json', 'r') as f:
    optimization_meta = json.load(f)
lgb_params = optimization_meta['lightgbm']['best_params']

print(f"\nTrain: {X_train.shape} | Val: {X_val.shape}")
print(f"Fraud rate: {y_train.mean()*100:.2f}%")
print(f"Month range: {month_train.min()} to {month_val.max()}")

Feature Selection (SHAP + Native Rank Sum <= 66):
  Original: 36 features
  Selected: 34 features
  Excluded: ['addr2_freq', 'card3_freq']

Train: (354324, 34) | Val: (118108, 34)
Fraud rate: 3.38%
Month range: 0 to 4

Train: (354324, 34) | Val: (118108, 34)
Fraud rate: 3.38%
Month range: 0 to 4


## 2. Pipeline ve Model

05_ModelOptimization'da LightGBM ile CatBoost'u 100 Optuna trial'da karşılaştırdık. LightGBM AUC 0.8800 ile baseline'ın (0.8766) üzerine çıkarken, CatBoost 0.8717'de kaldı. PR-AUC ve MCC'de de LightGBM önde - seçim net.

06_ModelEvaluation'da her feature için SHAP rank ve Native importance rank topladık. Threshold=66 ile `addr2_freq` ve `card3_freq` elendi - ikisi de hem SHAP hem native metrikte düşük sırada, modele anlamlı katkı sağlamıyor. Kalan 34 feature ile AUC 0.8842'ye ulaştık.

In [3]:
# Yeni feature set ile model eğit (Optuna parametreleriyle)
# lgb_params'dan class_weight varsa çıkar, biz kendi değerimizi kullanacağız
params_clean = {k: v for k, v in lgb_params.items() if k not in ['class_weight', 'random_state', 'verbose', 'n_jobs']}

model = LGBMClassifier(
    **params_clean,
    class_weight='balanced',
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Pipeline oluştur
pipeline = Pipeline([
    ('classifier', model)
])

print("Pipeline structure:")
for name, step in pipeline.named_steps.items():
    print(f"  {name}: {type(step).__name__}")
print(f"\nTrained on {X_train.shape[1]} features (threshold=66 selection)")
print(f"Model params: n_estimators={model.n_estimators}, max_depth={model.max_depth}")

Pipeline structure:
  classifier: LGBMClassifier

Trained on 34 features (threshold=66 selection)
Model params: n_estimators=751, max_depth=8


## 3. Eğitim ve Değerlendirme

In [4]:
# Model zaten 05'te train edildi, sadece tahmin yap
y_train_proba = pipeline.predict_proba(X_train)[:, 1]
y_val_proba = pipeline.predict_proba(X_val)[:, 1]
y_val_pred = pipeline.predict(X_val)

train_auc = roc_auc_score(y_train, y_train_proba)
val_auc = roc_auc_score(y_val, y_val_proba)

print(f"Using pre-trained optimized model from 05_ModelOptimization")
print(f"\nTrain AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f} | Gap: {train_auc - val_auc:.4f}")
print(f"Precision: {precision_score(y_val, y_val_pred):.4f} | Recall: {recall_score(y_val, y_val_pred):.4f} | F1: {f1_score(y_val, y_val_pred):.4f}")
print(f"\nExpected Val AUC: 0.8800 (from 05_ModelOptimization metadata)")

Using pre-trained optimized model from 05_ModelOptimization

Train AUC: 0.9892 | Val AUC: 0.8818 | Gap: 0.1074
Precision: 0.2482 | Recall: 0.6393 | F1: 0.3576

Expected Val AUC: 0.8800 (from 05_ModelOptimization metadata)


## 4. Cross-Validation (Time-Based)

05_ModelOptimization ile aynı strateji: GroupKFold with months. Fraud pattern'ları zamanla değişiyor ve gerçek dünyada model her zaman geçmiş veriyle eğitilip gelecek işlemleri tahmin edecek. Random split yapsak data leakage olur - Kasım verisinden öğrenip Ekim'i tahmin etmek gerçekçi değil.

In [5]:
# Combine train+val for CV (same as 05_ModelOptimization)
X_full = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_full = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)
month_full = np.concatenate([month_train, month_val])

# Time-based GroupKFold - same strategy as 05_ModelOptimization
cv = GroupKFold(n_splits=3)

print(f"CV Strategy: GroupKFold with {cv.n_splits} splits (month-based)")
print(f"Groups: month_num (prevents future data leakage)")

# Verify fold distribution
for i, (train_idx, val_idx) in enumerate(cv.split(X_full, y_full, groups=month_full)):
    train_months = np.unique(month_full[train_idx])
    val_months = np.unique(month_full[val_idx])
    print(f"  Fold {i+1}: Train months {train_months.tolist()} -> Val months {val_months.tolist()}")

# Manual CV (GroupKFold doesn't work with cross_val_score directly for groups)
cv_scores = []
for train_idx, val_idx in cv.split(X_full, y_full, groups=month_full):
    X_tr, X_vl = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_tr, y_vl = y_full.iloc[train_idx], y_full.iloc[val_idx]
    
    temp_model = LGBMClassifier(**{**lgb_params, 'class_weight': 'balanced', 'random_state': 42, 'verbose': -1, 'n_jobs': -1})
    temp_model.fit(X_tr, y_tr)
    cv_scores.append(roc_auc_score(y_vl, temp_model.predict_proba(X_vl)[:, 1]))

cv_scores = np.array(cv_scores)
print(f"\nCV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"Folds: {[f'{s:.4f}' for s in cv_scores]}")

CV Strategy: GroupKFold with 3 splits (month-based)
Groups: month_num (prevents future data leakage)
  Fold 1: Train months [1, 2, 3, 4] -> Val months [0]
  Fold 2: Train months [0, 1, 2] -> Val months [3, 4]
  Fold 3: Train months [0, 3, 4] -> Val months [1, 2]

CV AUC: 0.8586 (+/- 0.0151)
Folds: ['0.8378', '0.8648', '0.8732']

CV AUC: 0.8586 (+/- 0.0151)
Folds: ['0.8378', '0.8648', '0.8732']


In [6]:
# ============================================================
# BASELINE vs PIPELINE COMPARISON
# 03_Modeling (class_weight) vs 07_Pipeline (Optimized)
# ============================================================
print("=" * 80)
print("BASELINE (03_Modeling) vs PIPELINE (07_MLPipeline)")
print("=" * 80)

# 03_Modeling Results (class_weight version)
baseline_n_features_full = 392      # 03_Modeling full model - tüm feature'lar
baseline_n_features_interp = 16     # 03_Modeling interpretable model

baseline_full_auc = 0.9187
baseline_full_precision = 0.2814
baseline_full_recall = 0.7237
baseline_full_f1 = 0.4052

baseline_interp_auc = 0.8453
baseline_interp_precision = 0.1538
baseline_interp_recall = 0.6647
baseline_interp_f1 = 0.2498

# Pipeline metrics
pipeline_n_features = X_train.shape[1]
precision_val = precision_score(y_val, y_val_pred)
recall_val = recall_score(y_val, y_val_pred)
f1_val = f1_score(y_val, y_val_pred)

print(f"\n03_Modeling:")
print(f"  - Full Model: {baseline_n_features_full} feature (tüm original: V, id_, D, M, C dahil)")
print(f"  - Interpretable Model: {baseline_n_features_interp} feature (sadece yorumlanabilir)")
print(f"\n07_Pipeline (04_FE + 05_Optuna + 06_SHAP selection):")
print(f"  - Pipeline: {pipeline_n_features} feature (threshold=66 ile seçilmiş)")

print(f"\n{'Model':<40} {'Features':>8} {'AUC':>8} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 84)
print(f"{'03_Baseline Full (all features)':<40} {baseline_n_features_full:>8} {baseline_full_auc:>8.4f} {baseline_full_precision:>10.4f} {baseline_full_recall:>8.4f} {baseline_full_f1:>8.4f}")
print(f"{'03_Baseline Interp (yorumlanabilir)':<40} {baseline_n_features_interp:>8} {baseline_interp_auc:>8.4f} {baseline_interp_precision:>10.4f} {baseline_interp_recall:>8.4f} {baseline_interp_f1:>8.4f}")
print("-" * 84)
print(f"{'07_Pipeline (SHAP+Native threshold=66)':<40} {pipeline_n_features:>8} {val_auc:>8.4f} {precision_val:>10.4f} {recall_val:>8.4f} {f1_val:>8.4f}")

print(f"\n{'='*80}")
print("İYİLEŞTİRME ANALİZİ")
print(f"{'='*80}")

# Yüzde hesaplamaları
auc_pct = (val_auc - baseline_interp_auc) / baseline_interp_auc * 100
recall_pct = (recall_val - baseline_interp_recall) / baseline_interp_recall * 100
f1_pct = (f1_val - baseline_interp_f1) / baseline_interp_f1 * 100

print(f"\n03_Interp -> 07_Pipeline:")
print(f"  Features: {baseline_n_features_interp} -> {pipeline_n_features} ({pipeline_n_features - baseline_n_features_interp:+d} feature engineering ile)")
print(f"  AUC:      {baseline_interp_auc:.4f} -> {val_auc:.4f} ({val_auc - baseline_interp_auc:+.4f}, %{auc_pct:+.1f})")
print(f"  Recall:   {baseline_interp_recall:.4f} -> {recall_val:.4f} ({recall_val - baseline_interp_recall:+.4f}, %{recall_pct:+.1f})")
print(f"  F1:       {baseline_interp_f1:.4f} -> {f1_val:.4f} ({f1_val - baseline_interp_f1:+.4f}, %{f1_pct:+.1f})")

# Full vs Pipeline karşılaştırması
auc_full_pct = (val_auc - baseline_full_auc) / baseline_full_auc * 100

print(f"\n03_Full vs 07_Pipeline:")
print(f"  Features: {baseline_n_features_full} -> {pipeline_n_features} ({pipeline_n_features - baseline_n_features_full:+d})")
print(f"  AUC:      {baseline_full_auc:.4f} -> {val_auc:.4f} ({val_auc - baseline_full_auc:+.4f}, %{auc_full_pct:+.1f})")

print(f"\nSonuç: Interpretable {baseline_n_features_interp} değişkenden {pipeline_n_features} feature türettik")
print(f"       ve black-box V features olmadan bile Full baseline'a yakın performans elde ettik!")

BASELINE (03_Modeling) vs PIPELINE (07_MLPipeline)

03_Modeling:
  - Full Model: 392 feature (tüm original: V, id_, D, M, C dahil)
  - Interpretable Model: 16 feature (sadece yorumlanabilir)

07_Pipeline (04_FE + 05_Optuna + 06_SHAP selection):
  - Pipeline: 34 feature (threshold=66 ile seçilmiş)

Model                                    Features      AUC  Precision   Recall       F1
------------------------------------------------------------------------------------
03_Baseline Full (all features)               392   0.9187     0.2814   0.7237   0.4052
03_Baseline Interp (yorumlanabilir)            16   0.8453     0.1538   0.6647   0.2498
------------------------------------------------------------------------------------
07_Pipeline (SHAP+Native threshold=66)         34   0.8818     0.2482   0.6393   0.3576

İYİLEŞTİRME ANALİZİ

03_Interp -> 07_Pipeline:
  Features: 16 -> 34 (+18 feature engineering ile)
  AUC:      0.8453 -> 0.8818 (+0.0365, %+4.3)
  Recall:   0.6647 -> 0.6393 (-0.0

## 6. Pipeline Kaydetme

Model ve preprocessing tek pkl dosyasında. Production'da `joblib.load()` ile yükle, `predict_proba()` ile tahmin al. Feature sırası, model parametreleri, her şey kapsüllenmiş.

In [7]:
pipeline_dir = '../../models/fraud_detection/pipeline'
os.makedirs(pipeline_dir, exist_ok=True)

joblib.dump(pipeline, f'{pipeline_dir}/lgb_pipeline.pkl')

# Metadata
metadata = {
    'lgb_pipeline': {
        'val_auc': float(val_auc),
        'precision': float(precision_val),
        'recall': float(recall_val),
        'f1': float(f1_val),
        'n_features': int(X_train.shape[1]),
        'feature_columns': X_train.columns.tolist(),
        'excluded_features': excluded_features,
        'feature_selection': 'SHAP + Native Rank Sum <= 66',
        'source': '05_ModelOptimization params + 06_ModelEvaluation feature selection'
    },
    'model_selection_reason': 'LightGBM selected: AUC 0.8800 vs CatBoost 0.8717 in 05_ModelOptimization'
}

with open(f'{pipeline_dir}/pipeline_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved to:", pipeline_dir)
for f in os.listdir(pipeline_dir):
    print(f"  {f} ({os.path.getsize(f'{pipeline_dir}/{f}')/1024:.1f} KB)")

Saved to: ../../models/fraud_detection/pipeline
  advanced_xgb_pipeline.pkl (20923.9 KB)
  fraud_pipeline.pkl (19469.5 KB)
  lgb_pipeline.pkl (3656.1 KB)
  lgb_pipeline_lite.pkl (4043.9 KB)
  pipeline_metadata.json (1.3 KB)
  xgb_pipeline.pkl (19300.7 KB)
 ../../models/fraud_detection/pipeline
  advanced_xgb_pipeline.pkl (20923.9 KB)
  fraud_pipeline.pkl (19469.5 KB)
  lgb_pipeline.pkl (3656.1 KB)
  lgb_pipeline_lite.pkl (4043.9 KB)
  pipeline_metadata.json (1.3 KB)
  xgb_pipeline.pkl (19300.7 KB)


## 7. Yükleme Testi

In [8]:
loaded = joblib.load(f'{pipeline_dir}/lgb_pipeline.pkl')
loaded_auc = roc_auc_score(y_val, loaded.predict_proba(X_val)[:, 1])

print(f"Original: {val_auc:.4f} | Loaded: {loaded_auc:.4f} | Match: {'Yes' if abs(val_auc - loaded_auc) < 0.0001 else 'No'}")

sample_pred = loaded.predict_proba(X_val.iloc[[0]])[0, 1]
print(f"Sample prediction: {sample_pred:.4f} (actual: {y_val.iloc[0]})")

Original: 0.8818 | Loaded: 0.8818 | Match: Yes
Sample prediction: 0.0965 (actual: 0)


## 8. Production Inference

In [9]:
def predict_fraud(data, pipeline_path=None, threshold=0.5):
    if pipeline_path is None:
        pipeline_path = '../../models/fraud_detection/pipeline/lgb_pipeline.pkl'
    
    pipe = joblib.load(pipeline_path)
    proba = pipe.predict_proba(data)[:, 1]
    pred = (proba >= threshold).astype(int)
    risk = pd.cut(proba, bins=[0, 0.1, 0.3, 0.5, 0.7, 1.0],
                  labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
    
    return {'predictions': pred, 'probabilities': proba, 'risk_categories': risk}

# Test
results = predict_fraud(X_val.head(15))
print(pd.DataFrame({
    'Prob': results['probabilities'].round(3),
    'Pred': results['predictions'],
    'Risk': results['risk_categories'],
    'Actual': y_val.head(15).values
}).to_string(index=False))

 Prob  Pred      Risk  Actual
0.097     0  Very Low       0
0.075     0  Very Low       0
0.016     0  Very Low       0
0.068     0  Very Low       0
0.067     0  Very Low       0
0.029     0  Very Low       0
0.029     0  Very Low       0
0.413     0    Medium       0
0.030     0  Very Low       0
0.100     0       Low       0
0.802     1 Very High       1
0.021     0  Very Low       0
0.045     0  Very Low       0
0.002     0  Very Low       0
0.160     0       Low       0


## Sonuç

**Model seçimi:** 05_ModelOptimization'da 100 Optuna trial sonucu LightGBM (AUC 0.8800) CatBoost'u (0.8717) geçti. Eğitim süresi de daha kısa.

**Feature seçimi:** Orijinal 392 feature'dan 16 yorumlanabilir değişken seçtik, bunlardan 36 yeni feature türettik. SHAP + Native importance rank toplamı ile threshold=66 belirledik. `addr2_freq` ve `card3_freq` elendi, kalan 34 feature ile AUC 0.8842, Recall 0.6513 elde ettik.

**Production:** `lgb_pipeline.pkl` dosyası 34 feature, Optuna optimize parametreler ve SHAP feature selection içeriyor. `models/fraud_detection/pipeline/` dizininde deployment'a hazır.